<a href="https://colab.research.google.com/github/Keistkmiya/Tugas2-MachineLearning/blob/main/Tugas2_Chapter4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Chapter 4: Handling Numerical and Categorical Data

## Setup and Data Loading

In [1]:
import pandas as pd
import numpy as np
from sklearn import preprocessing

url_ev = "https://data.wa.gov/api/views/f6w7-q2d2/rows.csv?accessType=DOWNLOAD"
df_ev = pd.read_csv(url_ev)
df_ev.columns = [col.lower().replace(' ', '_') for col in df_ev.columns]

print("Setup Chapter 4 Berhasil!")

Setup Chapter 4 Berhasil!


## Rescaling a Feature
Dalam *machine learning*, kita sering kali berurusan dengan kolom numerik yang memiliki rentang nilai sangat berbeda. Misalnya, kolom 'model_year' memiliki nilai ribuan, sedangkan 'electric_range' bisa bernilai ratusan. Perbedaan skala ini bisa membingungkan algoritma tertentu.

Oleh karena itu, kita perlu melakukan **Rescaling** (sering disebut Min-Max Scaling) untuk membawa semua nilai ke dalam rentang yang sama, biasanya antara 0 dan 1. Teknik ini sangat penting untuk algoritma yang berbasis jarak seperti k-Nearest Neighbors (kNN) atau Neural Networks.

In [2]:
feature = df_ev[['electric_range']].values
minmax_scale = preprocessing.MinMaxScaler(feature_range=(0, 1))
scaled_feature = minmax_scale.fit_transform(feature)

print("5 Data Asli (Electric Range):")
print(feature[:5])

print("\n5 Data Setelah di-Rescale (0-1):")
print(scaled_feature[:5])

5 Data Asli (Electric Range):
[[ 75.]
 [308.]
 [215.]
 [  0.]
 [291.]]

5 Data Setelah di-Rescale (0-1):
[[0.22255193]
 [0.91394659]
 [0.6379822 ]
 [0.        ]
 [0.86350148]]


## Standardizing a Feature
Standardisasi (sering disebut *Z-score normalization*) adalah proses mengubah fitur sehingga memiliki rata-rata ($\mu$) bernilai 0 dan standar deviasi ($\sigma$) bernilai 1. Berbeda dengan *Min-Max Scaling* yang membatasi data dalam rentang tertentu (seperti 0-1), standardisasi tidak memiliki batas atas atau bawah yang kaku.

Keunggulan utama teknik ini adalah lebih tangguh terhadap pencilan (*outliers*) dibandingkan *Rescaling*. Rumus yang digunakan adalah:

$$z = \frac{x - \mu}{\sigma}$$

Kita biasanya menggunakan teknik ini sebagai standar awal sebelum melatih model seperti *Support Vector Machines* (SVM) atau *Logistic Regression*.

In [4]:
df_clean_4 = df_ev.dropna(subset=['electric_range']).copy()
x = df_clean_4[['electric_range']].values
standardizer = preprocessing.StandardScaler()
x_standardized = standardizer.fit_transform(x)

print(f"Rata-rata setelah standardisasi: {round(x_standardized.mean())}")
print(f"Standar deviasi setelah standardisasi: {x_standardized.std()}")
print("\n5 Data Pertama Hasil Standardisasi:")
print(x_standardized[:5])

Rata-rata setelah standardisasi: 0
Standar deviasi setelah standardisasi: 1.0

5 Data Pertama Hasil Standardisasi:
[[ 0.46521891]
 [ 3.45610085]
 [ 2.26231536]
 [-0.49751134]
 [ 3.237882  ]]


## Detecting Outliers

Mendeteksi pencilan adalah langkah penting untuk memastikan kualitas data kita. Salah satu metode yang paling umum dan efektif adalah menggunakan **Tukey's Method**, yang didasarkan pada *Interquartile Range* (IQR).

Kita mendefinisikan pencilan sebagai nilai yang berada di luar rentang:
1. Batas Bawah: $Q1 - 1.5 \times IQR$
2. Batas Atas: $Q3 + 1.5 \times IQR$

Di mana $Q1$ adalah kuartil pertama (persentil ke-25) dan $Q3$ adalah kuartil ketiga (persentil ke-75). Teknik ini sangat sering dibahas dalam literatur data science karena sifatnya yang objektif dan mudah diterapkan.

In [5]:
feature = df_clean_4['electric_range']
q1, q3 = np.percentile(feature, [25, 75])
iqr = q3 - q1
lower_bound = q1 - (1.5 * iqr)
upper_bound = q3 + (1.5 * iqr)
outliers = feature[(feature < lower_bound) | (feature > upper_bound)]

print(f"Batas Bawah: {lower_bound}")
print(f"Batas Atas: {upper_bound}")
print(f"Jumlah pencilan yang terdeteksi: {len(outliers)}")

if len(outliers) > 0:
    print("\nContoh data pencilan (10 teratas):")
    print(outliers.head(10))

Batas Bawah: -48.0
Batas Atas: 80.0
Jumlah pencilan yang terdeteksi: 43154

Contoh data pencilan (10 teratas):
1     308.0
2     215.0
4     291.0
6     200.0
8     215.0
9     210.0
10    259.0
12    259.0
16    322.0
17    215.0
Name: electric_range, dtype: float64


## Handling Outliers

Ada beberapa cara yang bisa kita lakukan untuk menangani pencilan setelah mereka terdeteksi:
1. **Menghapus (Dropping):** Kita membuang baris yang dianggap sebagai pencilan. Ini adalah cara termudah jika jumlah pencilan tidak terlalu banyak.
2. **Transformasi:** Kita bisa melakukan log transformation untuk mengecilkan variansi data.
3. **Capping (Winsorizing):** Kita mengganti nilai pencilan dengan nilai batas atas atau batas bawah yang telah kita tentukan sebelumnya.

Dalam langkah ini, kita akan memilih untuk menghapus pencilan agar model kita nantinya hanya belajar dari data yang berada dalam rentang normal. Kita akan menggunakan batas yang sudah kita hitung menggunakan metode IQR sebelumnya.

In [6]:
df_no_outliers = df_clean_4[(df_clean_4['electric_range'] >= lower_bound) &
                            (df_clean_4['electric_range'] <= upper_bound)]

print(f"Jumlah baris sebelum penanganan pencilan: {len(df_clean_4)}")
print(f"Jumlah baris setelah pencilan dihapus: {len(df_no_outliers)}")
print(f"Total baris yang dibuang: {len(df_clean_4) - len(df_no_outliers)}")
print("\nStatistik Jarak Tempuh Listrik setelah dibersihkan:")
print(df_no_outliers['electric_range'].describe())

Jumlah baris sebelum penanganan pencilan: 280821
Jumlah baris setelah pencilan dihapus: 237667
Total baris yang dibuang: 43154

Statistik Jarak Tempuh Listrik setelah dibersihkan:
count    237667.000000
mean          8.101882
std          15.960453
min           0.000000
25%           0.000000
50%           0.000000
75%           0.000000
max          76.000000
Name: electric_range, dtype: float64


## Encoding Nominal Categorical Features
Fitur nominal adalah kategori yang tidak memiliki urutan atau tingkatan tertentu (misalnya: merek mobil atau nama kota). Cara terbaik untuk menangani data seperti ini adalah dengan **One-Hot Encoding**.

Dalam proses ini, kita membuat kolom baru untuk setiap kategori yang ada, lalu memberikan nilai **1** jika baris tersebut termasuk dalam kategori tersebut, dan **0** jika tidak. Kita menggunakan fungsi `pd.get_dummies()` dari Pandas yang sangat praktis untuk tugas ini. Sesuai dengan pendekatan yang kita pelajari, teknik ini mencegah model menganggap bahwa satu kategori "lebih tinggi" nilainya daripada kategori lain hanya karena urutan angka.

In [8]:
print("Kategori dalam 'electric_vehicle_type' sebelum encoding:")
print(df_no_outliers['electric_vehicle_type'].unique())

df_encoded = pd.get_dummies(df_no_outliers, columns=['electric_vehicle_type'], drop_first=True)

print("\nHasil One-Hot Encoding (5 baris pertama):")
display(df_encoded.filter(like='electric_vehicle_type').head())

Kategori dalam 'electric_vehicle_type' sebelum encoding:
['Battery Electric Vehicle (BEV)' 'Plug-in Hybrid Electric Vehicle (PHEV)']

Hasil One-Hot Encoding (5 baris pertama):


,electric_vehicle_type_Plug-in Hybrid Electric Vehicle (PHEV)
0,False
3,False
5,True
7,True
11,True


## Encoding Ordinal Categorical Features

Untuk data ordinal, kita tidak boleh menggunakan *One-Hot Encoding* karena hal itu akan menghancurkan informasi tentang urutan data tersebut. Sebagai gantinya, kita menggunakan kamus pemetaan (*mapping dictionary*) untuk mengubah label teks menjadi angka yang mewakili urutannya (misalnya 1, 2, dan 3).

Karena di dataset EV ini tidak ada kolom ordinal bawaan yang jelas, kita akan membuat satu kolom buatan bernama `range_class` (kelas jarak tempuh: 'Short', 'Medium', 'Long') terlebih dahulu sebagai bahan latihan, lalu kita akan melakukan *encoding* pada kolom tersebut menggunakan metode `replace`.

In [10]:
def classify_range(r):
    if r > 200:
        return 'Long'
    elif r > 100:
        return 'Medium'
    else:
        return 'Short'

df_encoded['range_class'] = df_encoded['electric_range'].apply(classify_range)
ordinal_mapping = {
    'Short': 1,
    'Medium': 2,
    'Long': 3
}
df_encoded['range_class_encoded'] = df_encoded['range_class'].replace(ordinal_mapping).infer_objects(copy=False)
print("Data setelah Ordinal Encoding (Teks berubah jadi angka urut):")
display(df_encoded[['electric_range', 'range_class', 'range_class_encoded']].head())

Data setelah Ordinal Encoding (Teks berubah jadi angka urut):


/tmp/ipykernel_7180/1532235930.py:15: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_encoded['range_class_encoded'] = df_encoded['range_class'].replace(ordinal_mapping).infer_objects(copy=False)


,electric_range,range_class,range_class_encoded
0,75.0,Short,1
3,0.0,Short,1
5,72.0,Short,1
7,53.0,Short,1
11,18.0,Short,1


## Imputing Missing Class Values

Ketika kita memiliki data kategori yang hilang (Missing Values), cara paling sederhana adalah mengisinya dengan kategori yang paling sering muncul (modus). Namun, cara yang lebih "pintar" adalah dengan memprediksi nilai yang hilang tersebut menggunakan algoritma machine learning seperti **k-Nearest Neighbors (kNN)**.

kNN akan melihat baris data lain yang memiliki karakteristik serupa (misalnya jangkauan listrik yang mirip) lalu "menebak" kategori apa yang paling mungkin untuk data yang kosong tersebut. Teknik ini jauh lebih akurat daripada sekadar mengisi dengan nilai asal.

In [11]:
from sklearn.impute import KNNImputer

df_impute = df_encoded.copy()
df_impute.loc[0:5, 'range_class_encoded'] = np.nan

print(f"Jumlah data kosong yang kita buat: {df_impute['range_class_encoded'].isnull().sum()}")
imputer = KNNImputer(n_neighbors=2)
df_imputed_values = imputer.fit_transform(df_impute[['electric_range', 'range_class_encoded']])
df_impute['range_class_encoded'] = df_imputed_values[:, 1]

print(f"Jumlah data kosong setelah Imputing: {df_impute['range_class_encoded'].isnull().sum()}")
print("\nData yang tadi kosong sekarang sudah terisi (5 baris pertama):")
display(df_impute[['electric_range', 'range_class_encoded']].head())

Jumlah data kosong yang kita buat: 3
Jumlah data kosong setelah Imputing: 0

Data yang tadi kosong sekarang sudah terisi (5 baris pertama):


,electric_range,range_class_encoded
0,75.0,1.0
3,0.0,1.0
5,72.0,1.0
7,53.0,1.0
11,18.0,1.0
